# Fine-tune CLIP on Mars imagery (Colab)

Fine-tunes CLIP ViT-B/32 on MSL (24 classes) and/or HiRISE (8 classes) using
`finetune/finetune_clip.py` and `finetune/finetune_clip_hirise.py`, then benchmarks the
result with `eval/model_agnostic_retrieval_eval.py`.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).
Data, checkpoints and eval results are symlinked to Google Drive so they survive disconnects.

In [ ]:
!nvidia-smi

## 1. Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
%cd /content
if not os.path.isdir('mars-clip-bench'):
    !git clone https://github.com/anhkos/mars-clip-bench.git
%cd /content/mars-clip-bench
!git pull

## 2. Install dependencies
Colab already ships torch/torchvision, so this uses `requirements-colab.txt`.

In [ ]:
!pip install -q -r requirements-colab.txt

## 3. Point data / checkpoints / eval runs at Drive

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/mars-clip-bench-data"
for sub in ("data", "checkpoints", "eval_runs"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

!rm -rf data checkpoints eval/runs
!ln -s {DRIVE_ROOT}/data data
!ln -s {DRIVE_ROOT}/checkpoints checkpoints
!ln -s {DRIVE_ROOT}/eval_runs eval/runs

## 4. Download data (skipped automatically if already on Drive)

In [ ]:
!python scripts/download_data.py --dataset all
!python finetune/make_hirise_splits.py

## 5. Training config
Edit the values and run the cell. It rewrites the config constants at the top of the
finetune scripts (they have no CLI flags). Defaults match the repo's existing values.
Checkpoints are saved to Drive whenever val accuracy improves, but the scripts **do not
auto-resume**, so keep `EPOCHS` small enough to finish before the session ends.

In [ ]:
import re

EPOCHS       = 10
BATCH_SIZE   = 64      # drop to 32 if you hit CUDA OOM
LR           = 1e-6
WEIGHT_DECAY = 0.2
WARMUP_STEPS = 100

def patch(path):
    src = open(path).read()
    for name, val in dict(EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR,
                          WEIGHT_DECAY=WEIGHT_DECAY, WARMUP_STEPS=WARMUP_STEPS).items():
        src = re.sub(rf"^({name}\s*=\s*)\S+", rf"\g<1>{val!r}", src, flags=re.M)
    open(path, "w").write(src)

for p in ("finetune/finetune_clip.py", "finetune/finetune_clip_hirise.py"):
    patch(p)
!grep -nE "^(EPOCHS|BATCH_SIZE|LR|WEIGHT_DECAY|WARMUP_STEPS)" finetune/finetune_clip.py finetune/finetune_clip_hirise.py

## 6. Fine-tune
Run whichever you need. Each ends by re-embedding all images with the best checkpoint.

In [ ]:
# MSL -> checkpoints/clip_finetuned.pt
!python finetune/finetune_clip.py

In [ ]:
# HiRISE -> checkpoints/hirise_clip_finetuned.pt
!python finetune/finetune_clip_hirise.py

## 7. Benchmark the fine-tuned checkpoints

In [ ]:
!python eval/model_agnostic_retrieval_eval.py --model clip --model-id ViT-B/32 --checkpoint checkpoints/clip_finetuned.pt --dataset msl
!python eval/model_agnostic_retrieval_eval.py --model clip --model-id ViT-B/32 --checkpoint checkpoints/hirise_clip_finetuned.pt --dataset hirise

## 8. (Optional) Zero-shot baseline for comparison

In [ ]:
!python eval/model_agnostic_retrieval_eval.py --model clip --model-id ViT-B/32 --dataset msl
!python eval/model_agnostic_retrieval_eval.py --model clip --model-id ViT-B/32 --dataset hirise

Results are in `eval/runs/` and checkpoints in `checkpoints/`, both on Drive under `MyDrive/mars-clip-bench-data/`.